# Multi-Agent · Day 38 — **Capstone: Credit Analysis System**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~3 hrs

This is the module capstone — the artefact you'd actually walk a Barclays panel through. A **4-agent credit-analysis pipeline** on a real LangGraph graph, with a **human approval gate** before the memo is issued, built from everything in Days 31–37:

- **typed state + reducers** (Day 33) so agents can't hand each other junk,
- **the supervisor / staged graph** shape (Day 33),
- **shared results across hops** (Day 36),
- **retry + degrade** on the flaky external call (Day 37),
- **checkpointer + `interrupt_before`** for the human gate and crash-resume (Days 33, 37).

Pipeline: **Ingestion → Analysis → Compliance → [human gate] → Report.** Runnable on real LangGraph with an in-memory checkpointer; no API keys.

Today:
1. The **typed state** every agent reads/writes.
2. The **four agents**, each with its own access boundary + failure handling.
3. **Assemble the graph** with a human approval gate.
4. **Run it** — happy path, the pause-for-approval, resume, and a rejection path.
5. **Test suite** — 5 scenarios + E2E latency.

## 1. Typed state

One `TypedDict` is the contract for the whole run. `Annotated[list, operator.add]` reducers make the audit log and any list field *append* across hops instead of overwrite — the safe-concurrency pattern from Day 33.

In [1]:
from typing import Annotated, TypedDict, Literal
import operator, time

class CreditState(TypedDict):
    # inputs
    customer_id: str
    facility_amount: float
    # accumulated across hops
    log: Annotated[list[str], operator.add]      # append-only audit trail
    profile: dict          # ingestion writes
    analysis: dict         # analysis writes
    compliance: dict       # compliance writes
    confidence: float      # degraded steps lower this
    decision: str          # human sets / report reads
    report: str

def stamp(msg: str) -> dict:
    return {"log": [f"{time.strftime('%H:%M:%S')} {msg}"]}

☝ Every agent returns a **partial** state update (`{"profile": ...}`), never the whole dict — LangGraph merges it, and the `log` reducer appends rather than clobbers. That partial-update discipline is what lets you add or reorder agents later without any one of them needing to know the full shape. `confidence` is the Day-37 degradation signal threaded through state so the human gate can act on it.

## 2. The four agents

Each is a node with a **single responsibility and a hard access boundary** — only ingestion touches the customer store, only compliance holds the sanctions list, only report writes prose. Analysis has the flaky external scoring call, wrapped in retry-then-degrade (Day 37).

In [2]:
import random

CUSTOMERS = {
    "C-4417": {"name": "ACME Ltd", "turnover": 4_200_000, "arrears": 0, "counterparty": "ACME Ltd"},
    "C-9001": {"name": "Vulpes SA", "turnover": 90_000, "arrears": 2, "counterparty": "Vulpes SA"},
}
SANCTIONS = {"Vulpes SA"}

def ingestion_agent(state: CreditState) -> dict:
    cid = state["customer_id"]
    if cid not in CUSTOMERS:
        raise ValueError(f"unknown customer {cid}")          # required step -> hard fail
    return {"profile": CUSTOMERS[cid], **stamp(f"ingested {cid}")}

def _flaky_score(_c={"n": 0}):
    _c["n"] += 1
    if _c["n"] % 3 != 0:                                       # fails ~2 of 3 calls
        raise TimeoutError("scoring service 503")
    return None

def analysis_agent(state: CreditState) -> dict:
    p = state["profile"]
    # retry the flaky external scorer up to 3x, then DEGRADE (don't crash the memo)
    conf = 1.0
    for attempt in range(3):
        try:
            _flaky_score(); break
        except TimeoutError:
            if attempt == 2:
                conf = 0.5                                    # degraded: heuristic fallback
    # simple, deterministic scoring heuristic
    pd = 0.01 + 0.05 * p["arrears"] + (0.04 if p["turnover"] < 250_000 else 0.0)
    grade = "AAA" if pd < 0.02 else "BBB" if pd < 0.08 else "CCC"
    limit = round(min(state["facility_amount"], p["turnover"] * 0.2))
    return {"analysis": {"pd": round(pd, 3), "grade": grade, "suggested_limit": limit},
            "confidence": conf,
            **stamp(f"scored pd={pd:.3f} grade={grade} conf={conf:.0%}")}

def compliance_agent(state: CreditState) -> dict:
    party = state["profile"]["counterparty"]
    hit = party in SANCTIONS
    return {"compliance": {"counterparty": party, "sanctions_hit": hit},
            **stamp(f"compliance {'HIT' if hit else 'clear'} for {party}")}

☝ The boundaries are **structural** (Day 33): `analysis_agent` has no reference to `SANCTIONS`, `compliance_agent` has none to `CUSTOMERS` — an auditor verifies separation of duties by reading which globals each function closes over, not by trusting a design doc. And note analysis never crashes on the flaky scorer: three retries, then it drops `confidence` to 0.5 and falls back to a heuristic. A degraded memo that says so beats no memo (Day 37).

## 3. Assemble the graph — with the human gate

The routing rule: a **sanctions hit** or **low confidence** must not auto-issue a memo — it pauses for a human. We express the gate with `interrupt_before=["report"]` on a compiled graph that carries a checkpointer, so the run parks *before* writing the memo and a person resumes it.

In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

def decision_gate(state: CreditState) -> dict:
    """Set a provisional decision; the human confirms/overrides at the interrupt."""
    hit = state["compliance"]["sanctions_hit"]
    low = state["confidence"] < 0.6
    provisional = "REFER" if (hit or low) else "APPROVE"
    reason = "sanctions hit" if hit else "low confidence" if low else "clean"
    return {"decision": provisional, **stamp(f"gate -> {provisional} ({reason})")}

def report_agent(state: CreditState) -> dict:
    a, c = state["analysis"], state["compliance"]
    memo = (f"CREDIT MEMO — {state['profile']['name']} ({state['customer_id']})\n"
            f"  Grade {a['grade']}  PD {a['pd']:.1%}  Limit GBP {a['suggested_limit']:,}\n"
            f"  Compliance: {'FLAGGED' if c['sanctions_hit'] else 'clear'}\n"
            f"  Confidence: {state['confidence']:.0%}\n"
            f"  DECISION: {state['decision']}")
    return {"report": memo, **stamp("memo issued")}

builder = StateGraph(CreditState)
for name, fn in [("ingestion", ingestion_agent), ("analysis", analysis_agent),
                 ("compliance", compliance_agent), ("gate", decision_gate),
                 ("report", report_agent)]:
    builder.add_node(name, fn)
builder.add_edge(START, "ingestion")
builder.add_edge("ingestion", "analysis")
builder.add_edge("analysis", "compliance")
builder.add_edge("compliance", "gate")
builder.add_edge("gate", "report")
builder.add_edge("report", END)

# checkpointer + interrupt_before = crash-resume AND the human approval gate
app = builder.compile(checkpointer=MemorySaver(), interrupt_before=["report"])
print("graph compiled — nodes:", list(app.get_graph().nodes)[1:])

graph compiled — nodes: ['ingestion', 'analysis', 'compliance', 'gate', 'report', '__end__']


☝ One `compile` call buys two Day-37 properties at once. `interrupt_before=["report"]` is the **human gate** — the graph stops before issuing the memo and hands control back. The `MemorySaver` checkpointer is what makes that pause *durable*: state is persisted at every node, so the same mechanism that lets a human resume also lets a **crashed orchestrator resume** from the last checkpoint. In production `MemorySaver` becomes `PostgresSaver`; nothing else changes.

## 4. Run it — pause, inspect, resume

A run drives to the interrupt, a human inspects state, then resumes (optionally overriding the decision). `thread_id` names the run so its checkpoints are isolated.

In [4]:
def run_to_gate(customer_id: str, amount: float, thread: str):
    cfg = {"configurable": {"thread_id": thread}}
    app.invoke({"customer_id": customer_id, "facility_amount": amount,
                "log": [], "confidence": 1.0}, cfg)
    snap = app.get_state(cfg)
    return cfg, snap

# --- clean customer: gate says APPROVE, human confirms ---
cfg, snap = run_to_gate("C-4417", 500_000, thread="run-clean")
print("PAUSED before:", snap.next, "| provisional:", snap.values["decision"])
final = app.invoke(None, cfg)                          # human approves as-is -> resume
print(final["report"])

PAUSED before: ('report',) | provisional: APPROVE
CREDIT MEMO — ACME Ltd (C-4417)
  Grade AAA  PD 1.0%  Limit GBP 500,000
  Compliance: clear
  Confidence: 100%
  DECISION: APPROVE


In [5]:
# --- sanctioned customer: gate says REFER; human OVERRIDES to REJECT before resuming ---
cfg, snap = run_to_gate("C-9001", 500_000, thread="run-flagged")
print("PAUSED before:", snap.next, "| provisional:", snap.values["decision"],
      "| sanctions_hit:", snap.values["compliance"]["sanctions_hit"])
app.update_state(cfg, {"decision": "REJECT", "log": ["human override -> REJECT"]})
final = app.invoke(None, cfg)
print(final["report"])

PAUSED before: ('report',) | provisional: REFER | sanctions_hit: True
CREDIT MEMO — Vulpes SA (C-9001)
  Grade CCC  PD 15.0%  Limit GBP 18,000
  Compliance: FLAGGED
  Confidence: 100%
  DECISION: REJECT


☝ This is the whole point of a human gate in a regulated flow: the machine **proposes** (`REFER`), a person **disposes** (`update_state(... decision=REJECT)`), and the override is itself written to the audit `log`. Because the resume is `invoke(None, cfg)`, the report agent runs against the human-corrected state — the memo reflects the human's decision, not the model's guess. Every step, every override, every confidence drop is in `log`, reconstructable from the checkpoint months later.

## 5. Test suite — 5 scenarios + E2E latency

A capstone isn't done until it's *tested*. Five scenarios cover the branches: clean approve, sanctioned refer, tiny-turnover degrade, unknown-customer hard-fail, large-facility limit cap. We assert the gate's provisional decision and measure end-to-end latency.

In [6]:
import uuid

SCENARIOS = [
    # (customer, amount, expect_decision, note)
    ("C-4417", 500_000, "APPROVE", "clean SME"),
    ("C-9001", 500_000, "REFER",   "sanctions hit -> human"),
    ("C-4417", 5_000_000, "APPROVE", "limit capped at 20% turnover"),
    ("C-9001", 10_000,  "REFER",   "sanctioned + tiny -> human"),
    ("C-0000", 100_000, "ERROR",   "unknown customer -> hard fail"),
]

def gate_decision(customer, amount):
    cfg = {"configurable": {"thread_id": f"test-{uuid.uuid4().hex[:6]}"}}
    app.invoke({"customer_id": customer, "facility_amount": amount,
                "log": [], "confidence": 1.0}, cfg)
    return app.get_state(cfg).values["decision"]

passed, latencies = 0, []
for cust, amt, expect, note in SCENARIOS:
    t0 = time.perf_counter()
    try:
        got = gate_decision(cust, amt)
    except Exception:
        got = "ERROR"
    dt = (time.perf_counter() - t0) * 1000
    latencies.append(dt)
    ok = got == expect
    passed += ok
    print(f"  [{'PASS' if ok else 'FAIL'}] {note:32} expect={expect:8} got={got} ({dt:.0f}ms)")

print(f"\n{passed}/{len(SCENARIOS)} passed | "
      f"E2E latency p50~{sorted(latencies)[len(latencies)//2]:.0f}ms")

  [PASS] clean SME                        expect=APPROVE  got=APPROVE (1ms)
  [PASS] sanctions hit -> human           expect=REFER    got=REFER (1ms)
  [PASS] limit capped at 20% turnover     expect=APPROVE  got=APPROVE (1ms)
  [PASS] sanctioned + tiny -> human       expect=REFER    got=REFER (1ms)
  [PASS] unknown customer -> hard fail    expect=ERROR    got=ERROR (0ms)

5/5 passed | E2E latency p50~1ms


☝ The suite pins **behaviour at the gate**, not prose — deterministic, so it belongs in CI (Day 29's quality gates). Each scenario hits a distinct branch: the limit-cap case proves ACME's £5m ask is capped to 20% of turnover, the unknown-customer case proves a *required* step fails hard rather than degrading. Latency here is trivial because the LLM calls are stubbed; the *shape* is what you keep — in production you'd assert p50/p95 against an SLA and alert on regressions.

## 6. Recap — the Module 5 stack, assembled

| Capstone element | From | Cell |
|---|---|---|
| Typed state + append-only audit `log` | Day 33 | 1 |
| 4 agents, structural access boundaries | Days 32–33 | 2 |
| Retry-then-degrade on flaky call | Day 37 | 2 |
| Staged graph on real LangGraph | Day 33 | 3 |
| Human approval gate (`interrupt_before`) | Days 33, 37 | 3 |
| Crash-resume (checkpointer + `thread_id`) | Days 33, 37 | 3–4 |
| Human override written to audit trail | Day 37 | 4 |
| Deterministic scenario tests + latency | Day 29 | 5 |

**What to say in the demo:** "It's a graph, so every path is drawable and auditable; it checkpoints, so it survives a crash and pauses for a human; and it degrades honestly instead of failing, with a confidence signal the gate acts on." That's the AVP-level answer.

**Module 5 complete.** Next (Day 39, Module 6): agent **memory** — the four memory types and how an agent remembers a customer across sessions.